In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os,sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.optim as optim
from torch.utils.data import Dataset
import kagglehub
from pathlib import Path
from PIL import Image
from torchvision import transforms


from UNet import UNet, UNetTrainer
from data import CarvanaDataset

In [ ]:
# Carvana

BATCH_SIZE = 1  
NUM_WORKERS = 4  # threads 2~8

data_root = "../../Data/Carvana_segmentation" 

path = kagglehub.competition_download(
    'carvana-image-masking-challenge', 
    output_dir=data_root
)

print(f"Dataset downloaded: {path}")

dir_img = Path('./data/imgs/')
dir_mask = Path('./data/masks/')

train_set = CarvanaDataset(dir_img, dir_mask)

train_loader = torch.utils.data.DataLoader(
    dataset=train_set,        
    batch_size=BATCH_SIZE,    
    shuffle=True,             
    num_workers=NUM_WORKERS,  
    pin_memory=True,          # speeds up CPU-GPU batch transfer
    drop_last=False           
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_set,         
    batch_size=BATCH_SIZE,    
    shuffle=False,            
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

In [ ]:
model_save_root = "../../Models/UNet"
os.makedirs(model_save_root)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# UNet

#### Train

In [ ]:
net=UNet(num_class=10)

lr=1e-5

optimizer = optim.Adam(net.parameters(), lr=lr)

trainer=UNetTrainer(model=net,
                       optimizer=optimizer,
                       dtype=float,
                       device=device)

trainer.train(num_epochs=5,
              train_dataloader=train_loader,
              log_interval=50)

torch.save(trainer.model.state_dict(),os.path.join(model_save_root,"UNet.pth"))

#### Evaluation

In [ ]:
net=UNet(num_class=10).to(device)
net.load_state_dict(torch.load(os.path.join(model_save_root,"UNet.pth"), map_location=device))
net.eval()


with torch.no_grad():   
    for batch in tqdm(dataloader, total=num_val_batches, unit='batch', leave=False):
        image, mask_true = batch['image'], batch['mask']

        # move images and labels to correct device and type
        image = image.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
        mask_true = mask_true.to(device=device, dtype=torch.long)

        # predict the mask
         mask_pred = net(image)

            if net.n_classes == 1:
                assert mask_true.min() >= 0 and mask_true.max() <= 1, 'True mask indices should be in [0, 1]'
                mask_pred = (F.sigmoid(mask_pred) > 0.5).float()
                # compute the Dice score
                dice_score += dice_coeff(mask_pred, mask_true, reduce_batch_first=False)
            else:
                assert mask_true.min() >= 0 and mask_true.max() < net.n_classes, 'True mask indices should be in [0, n_classes['
                # convert to one-hot format
                mask_true = F.one_hot(mask_true, net.n_classes).permute(0, 3, 1, 2).float()
                mask_pred = F.one_hot(mask_pred.argmax(dim=1), net.n_classes).permute(0, 3, 1, 2).float()
                # compute the Dice score, ignoring background
                dice_score += multiclass_dice_coeff(mask_pred[:, 1:], mask_true[:, 1:], reduce_batch_first=False)

    net.train()
    return dice_score / max(num_val_batches, 1)